# Mikado Synthetic Data Generator

Generates synthetic Mikado training images using Blender's rigid-body physics.
Sticks are dropped into a pile, rendered from above, and YOLO-OBB labels are exported automatically.

**Runtime → Change runtime type → T4 GPU** (optional — speeds up Cycles rendering)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURE THESE
# ---------------------------------------------------------------------------
DRIVE_ROOT = '/content/drive/MyDrive/Data/mikado-judge'

# Where to save the generated dataset
OUTPUT_DIR = f'{DRIVE_ROOT}/synthetic'

# Number of images to generate
COUNT = 150

# Random seed (0 = different each run)
SEED = 0

# Stick material style: "simple" (flat color) or "realistic" (wood grain + ring tips)
STICK_STYLE = "simple"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output:      {OUTPUT_DIR}')
print(f'Count:       {COUNT}')
print(f'Stick style: {STICK_STYLE}')

## 2. Install Blender

In [ ]:
# Install Blender (headless)
!apt-get install -y blender 2>&1 | tail -3
!/usr/bin/blender --version 2>&1 | head -1
BLENDER = '/usr/bin/blender'
print(f'Blender path: {BLENDER}')

# Cache Cycles CUDA kernels to Drive so compilation only happens once ever.
# First run: compiles (~5 min). All subsequent runs: loads from cache (~5 sec).
import os
KERNEL_CACHE_DIR = f'{DRIVE_ROOT}/blender_cycles_cache'
os.makedirs(KERNEL_CACHE_DIR, exist_ok=True)
os.environ['CYCLES_KERNEL_PATH'] = KERNEL_CACHE_DIR
print(f'Cycles kernel cache: {KERNEL_CACHE_DIR}')
cache_files = list(__import__('pathlib').Path(KERNEL_CACHE_DIR).glob('*.cubin'))
print(f'Cached kernels found: {len(cache_files)} (0 = will compile on first render)')

In [ ]:
# Install pyyaml for Blender's Python
import subprocess

# Install pyyaml via apt (works regardless of pip availability)
r = subprocess.run(['apt-get', 'install', '-y', 'python3-yaml'], capture_output=True, text=True)
print('python3-yaml installed.' if r.returncode == 0 else r.stderr[-300:])

# Verify yaml is importable by Blender
r2 = subprocess.run(
    ['/usr/bin/blender', '--background', '--python-expr', 'import yaml; print("yaml OK:", yaml.__version__)'],
    capture_output=True, text=True
)
yaml_lines = [l for l in r2.stdout.splitlines() if 'yaml' in l.lower()]
print(yaml_lines[0] if yaml_lines else 'WARNING: yaml not found in Blender Python')

## 3. Clone repository

In [ ]:
REPO_DIR = '/content/mikado-blender'
REPO_URL = 'https://github.com/spyszkowski/mikado-blender.git'

import os
if os.path.exists(REPO_DIR):
    print('Repository exists, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    print('Cloning...')
    !git clone --branch master {REPO_URL} {REPO_DIR}

print('Done.')
!find {REPO_DIR} -type f

## 3b. Clean & update (re-run to refresh)

In [ ]:
# ---------------------------------------------------------------------------
# CLEAN & UPDATE  — run this cell to wipe old output and pull latest code
# ---------------------------------------------------------------------------
import glob, os

# 1. Pull latest scripts/configs from GitHub
!cd {REPO_DIR} && git pull

# 2. Wipe previous synthetic output so old images don't mix with new ones
#    Remove contents only (not the directory itself) to avoid Google Drive
#    moving the folder to Trash and confusing the UI.
for subdir in ('images', 'labels'):
    d = os.path.join(OUTPUT_DIR, subdir)
    if os.path.isdir(d):
        for f in glob.glob(os.path.join(d, '*')):
            os.remove(f)
        print(f'Cleared: {d}')
os.makedirs(f'{OUTPUT_DIR}/images', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/labels', exist_ok=True)
print('Output directory ready.')

## 4. Generate synthetic images

In [ ]:
import subprocess, os

KERNEL_CACHE_DIR = f'{DRIVE_ROOT}/blender_cycles_cache'
os.makedirs(KERNEL_CACHE_DIR, exist_ok=True)

cmd = [
    '/usr/bin/blender', '--background',
    '--python', f'{REPO_DIR}/scripts/generate.py', '--',
    '--count', str(COUNT),
    '--output', OUTPUT_DIR,
    '--config', f'{REPO_DIR}/configs',
    '--seed', str(SEED),
    '--stick-style', STICK_STYLE,
]

env = os.environ.copy()
env['CYCLES_KERNEL_PATH'] = KERNEL_CACHE_DIR

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, env=env, capture_output=True, text=True)
# Show last 3000 chars of stdout/stderr for debugging
if result.stdout:
    print('--- STDOUT (last 3000 chars) ---')
    print(result.stdout[-3000:])
if result.stderr:
    print('--- STDERR (last 3000 chars) ---')
    print(result.stderr[-3000:])
print('Exit code:', result.returncode)

## 5. Verify output

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image, ImageDraw
import random

images = sorted(Path(OUTPUT_DIR, 'images').glob('*.png'))
labels = sorted(Path(OUTPUT_DIR, 'labels').glob('*.txt'))
print(f'Generated: {len(images)} images, {len(labels)} label files')

# Count total annotations
total_sticks = sum(len(l.read_text().strip().splitlines()) for l in labels if l.stat().st_size > 0)
print(f'Total stick annotations: {total_sticks}')
if images:
    print(f'Avg sticks per image: {total_sticks / len(images):.1f}')

In [ ]:
# Show 4 sample images with OBB overlays
CLASS_COLORS = {
    0: (180, 0, 180),   # mikado — purple
    1: (30, 80, 230),   # blue
    2: (230, 30, 30),   # red
    3: (230, 200, 30),  # yellow
    4: (30, 180, 60),   # green
}

samples = random.sample(list(zip(images, labels)), min(4, len(images)))
for img_path, lbl_path in samples:
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    draw = ImageDraw.Draw(img)
    for line in lbl_path.read_text().strip().splitlines():
        parts = list(map(float, line.split()))
        cid = int(parts[0])
        pts = [(parts[i]*w, parts[i+1]*h) for i in range(1, 9, 2)]
        color = CLASS_COLORS.get(cid, (255, 255, 255))
        draw.polygon(pts, outline=color)
    img = img.resize((min(w, 1200), int(h * min(w, 1200) / w)))
    print(img_path.name)
    display(img)

## 6. Merge synthetic into mikado-judge dataset

Copies synthetic images + labels into the **train** split of the real dataset.
Validation stays real-only so metrics reflect real-world performance.

In [ ]:
# ---------------------------------------------------------------------------
# Merge synthetic images into the mikado-judge YOLO dataset (train split only)
# ---------------------------------------------------------------------------
import shutil, os
from pathlib import Path

# Path to the mikado-judge YOLO dataset (prepared by prepare_dataset.py)
DATASET_DIR = f'{DRIVE_ROOT}/datasets/mikado'

synth_images = sorted(Path(OUTPUT_DIR, 'images').glob('*.png'))
synth_labels = sorted(Path(OUTPUT_DIR, 'labels').glob('*.txt'))

if not Path(DATASET_DIR, 'images', 'train').exists():
    print(f'ERROR: Real dataset not found at {DATASET_DIR}')
    print('Run mikado-judge/scripts/prepare_dataset.py first to create the base dataset.')
else:
    train_img_dir = Path(DATASET_DIR, 'images', 'train')
    train_lbl_dir = Path(DATASET_DIR, 'labels', 'train')

    # Count existing real images (before merge)
    existing = set(p.name for p in train_img_dir.iterdir())
    real_count = len(existing)

    # Remove old synthetic files from previous merges
    old_synth = [p for p in train_img_dir.glob('synthetic_*')]
    for p in old_synth:
        p.unlink()
    old_synth_lbl = [p for p in train_lbl_dir.glob('synthetic_*')]
    for p in old_synth_lbl:
        p.unlink()
    if old_synth:
        print(f'Cleaned {len(old_synth)} old synthetic files from previous merge.')

    # Copy new synthetic data
    copied = 0
    for img_path in synth_images:
        lbl_path = Path(OUTPUT_DIR, 'labels', img_path.stem + '.txt')
        if lbl_path.exists():
            shutil.copy2(img_path, train_img_dir / img_path.name)
            shutil.copy2(lbl_path, train_lbl_dir / lbl_path.name)
            copied += 1

    # Stats
    real_now = len([p for p in train_img_dir.iterdir() if not p.name.startswith('synthetic_')])
    synth_now = len([p for p in train_img_dir.glob('synthetic_*')])
    val_count = len(list(Path(DATASET_DIR, 'images', 'val').iterdir()))

    print(f'\nMerge complete: {DATASET_DIR}')
    print(f'  Train: {real_now} real + {synth_now} synthetic = {real_now + synth_now} total')
    print(f'  Val:   {val_count} real (unchanged)')
    print(f'\nDataset ready for training with mikado-judge.')